# Logistic regression / Hồi quy logistic

Tôi lưu bài này trong nhật ký học để ôn lại cách viết và đọc code.

**Nguồn / Source:** Bản Colab `Buoi_04.Multi-class_Classification_BaiTap.ipynb`. Các bài có tên “Bản sao” là tài liệu thực hành tham khảo; tên file không được xem là bằng chứng tác giả.

**Cách dùng / Run:** chạy các cell theo thứ tự. Thư viện được dùng: `matplotlib.pyplot`, `numpy`, `pandas`, `seaborn`, `sklearn.linear_model`, `sklearn.metrics`, `sklearn.model_selection`, `sklearn.multiclass`, `sklearn.preprocessing`, `warnings`.

**Execution:** ARCHIVED_SOURCE — bản mã học tập, chưa ghi nhận chạy lại đầy đủ trong đợt này. Output lịch sử được bỏ; hướng dẫn và lỗi cần xử lý nằm trong bảng kỹ thuật.

Learning archive: coursework/tutorial practice, with original code retained for study. Original tutorial references embedded in the source are retained below.

#  Multi-class Classification


Estimated time needed: **30** mins


In this lab, you will learn the different strategies of Multi-class classification and implement the same on a real-world dataset.


## **Objectives**


Sau khi hoàn thành sinh viên có thể


1. Hiểu cách sử dụng mã hóa one-hot (one-hot encoding) cho các biến phân loại.
2. Triển khai hồi quy logistic cho bài toán phân loại đa lớp bằng các chiến lược One-vs-All (OvA) và One-vs-One (OvO).

3. Đánh giá hiệu suất mô hình bằng các chỉ số phù hợp.


## Import Necessary Libraries


First, to ensure the availability of the required libraries, execute the cell below.


In [ ]:
!pip install numpy==2.2.0
!pip install pandas==2.2.2
!pip install scikit-learn==1.6.0
!pip install matplotlib==3.9.3
!pip install seaborn==0.13.2

import libraries for data processing, model training, and evaluation.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsOneClassifier
from sklearn.metrics import accuracy_score

import warnings
warnings.filterwarnings('ignore')

## dataset
The data set being used for this lab is the "Obesity Risk Prediction" data set publically available on <a href="https://archive.ics.uci.edu/dataset/544/estimation+of+obesity+levels+based+on+eating+habits+and+physical+condition">UCI Library</a> under the <a href="https://creativecommons.org/licenses/by/4.0/legalcode">CCA 4.0</a> license. Dataset gồm 17 thuộc tính với tổng 2,111 mẫu.
Thuộc tính được trình bày bảng bên dưới


<style type="text/css">
.tg  {border-collapse:collapse;border-spacing:0;}
.tg td{border-color:black;border-style:solid;border-width:1px;font-family:Arial, sans-serif;font-size:14px;
  overflow:hidden;padding:10px 5px;word-break:normal;}
.tg th{border-color:black;border-style:solid;border-width:1px;font-family:Arial, sans-serif;font-size:14px;
  font-weight:normal;overflow:hidden;padding:10px 5px;word-break:normal;}
.tg .tg-7zrl{text-align:left;vertical-align:bottom}
</style>
<table class="tg"><thead>
  <tr>
    <th class="tg-7zrl">Variable Name</th>
    <th class="tg-7zrl">Type</th>
    <th class="tg-7zrl">Description</th>
  </tr></thead>
<tbody>
  <tr>
    <td class="tg-7zrl">Gender</td>
    <td class="tg-7zrl">Categorical</td>
    <td class="tg-7zrl"></td>
  </tr>
  <tr>
    <td class="tg-7zrl">Age</td>
    <td class="tg-7zrl">Continuous</td>
    <td class="tg-7zrl"></td>
  </tr>
  <tr>
    <td class="tg-7zrl">Height</td>
    <td class="tg-7zrl">Continuous</td>
    <td class="tg-7zrl"></td>
  </tr>
  <tr>
    <td class="tg-7zrl">Weight</td>
    <td class="tg-7zrl">Continuous</td>
    <td class="tg-7zrl"></td>
  </tr>
  <tr>
    <td class="tg-7zrl">family_history_with_overweight</td>
    <td class="tg-7zrl">Binary</td>
    <td class="tg-7zrl">Has a family member suffered or suffers from overweight?</td>
  </tr>
  <tr>
    <td class="tg-7zrl">FAVC</td>
    <td class="tg-7zrl">Binary</td>
    <td class="tg-7zrl">Do you eat high caloric food frequently?</td>
  </tr>
  <tr>
    <td class="tg-7zrl">FCVC</td>
    <td class="tg-7zrl">Integer</td>
    <td class="tg-7zrl">Do you usually eat vegetables in your meals?</td>
  </tr>
  <tr>
    <td class="tg-7zrl">NCP</td>
    <td class="tg-7zrl">Continuous</td>
    <td class="tg-7zrl">How many main meals do you have daily?</td>
  </tr>
  <tr>
    <td class="tg-7zrl">CAEC</td>
    <td class="tg-7zrl">Categorical</td>
    <td class="tg-7zrl">Do you eat any food between meals?</td>
  </tr>
  <tr>
    <td class="tg-7zrl">SMOKE</td>
    <td class="tg-7zrl">Binary</td>
    <td class="tg-7zrl">Do you smoke?</td>
  </tr>
  <tr>
    <td class="tg-7zrl">CH2O</td>
    <td class="tg-7zrl">Continuous</td>
    <td class="tg-7zrl">How much water do you drink daily?</td>
  </tr>
  <tr>
    <td class="tg-7zrl">SCC</td>
    <td class="tg-7zrl">Binary</td>
    <td class="tg-7zrl">Do you monitor the calories you eat daily?</td>
  </tr>
  <tr>
    <td class="tg-7zrl">FAF</td>
    <td class="tg-7zrl">Continuous</td>
    <td class="tg-7zrl">How often do you have physical activity?</td>
  </tr>
  <tr>
    <td class="tg-7zrl">TUE</td>
    <td class="tg-7zrl">Integer</td>
    <td class="tg-7zrl">How much time do you use technological devices such as cell phone, videogames, television, computer and others?</td>
  </tr>
  <tr>
    <td class="tg-7zrl">CALC</td>
    <td class="tg-7zrl">Categorical</td>
    <td class="tg-7zrl">How often do you drink alcohol?</td>
  </tr>
  <tr>
    <td class="tg-7zrl">MTRANS</td>
    <td class="tg-7zrl">Categorical</td>
    <td class="tg-7zrl">Which transportation do you usually use?</td>
  </tr>
  <tr>
    <td class="tg-7zrl">NObeyesdad</td>
    <td class="tg-7zrl">Categorical</td>
    <td class="tg-7zrl">Obesity level</td>
  </tr>
</tbody></table>


### Load the dataset

Load the data


In [ ]:
file_path = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/GkDzb7bWrtvGXdPOfk6CIg/Obesity-level-prediction-dataset.csv"
data = pd.read_csv(file_path)
data.head()

## Exploratory Data Analysis


Trực quan hóa phân bố của biến mục tiêu để hiểu mức độ cân bằng giữa các lớp


In [ ]:
# Distribution of target variable
sns.countplot(y='NObeyesdad', data=data)
plt.title('Distribution of Obesity Levels')
plt.show()

Điều này cho thấy bộ dữ liệu khá cân bằng và không cần xử lý đặc biệt liên quan đến việc đào tạo bị lệch.


### Exercise 1
Kiểm tra null values, và hiển thị tóm tắt thông tin về dataset (use `.info()` and `.describe()` methods).


In [ ]:
# your code here

print(data.info())
print(data.describe())

<details><summary>Click here for the solution</summary>

```python
# Checking for null values
print(data.isnull().sum())

# Dataset summary
print(data.info())
print(data.describe())
```
</details>


Kết quả mong đợi:

Số lượng giá trị null của từng cột (khả năng cao là bằng 0 đối với bộ dữ liệu này).

Thông tin về bộ dữ liệu, bao gồm tên cột, kiểu dữ liệu và mức sử dụng bộ nhớ.

Thống kê mô tả cho các cột dạng số.


## Preprocessing the data


### Feature scaling
Chuẩn hóa các đặc trưng số để đưa chúng về cùng thang đo, giúp mô hình hoạt động hiệu quả hơn


In [ ]:
# Chuẩn hóa các đặc trưng số liên tục
continuous_columns = data.select_dtypes(include=['float64']).columns.tolist()

scaler = StandardScaler()
scaled_features = scaler.fit_transform(data[continuous_columns])

# Chuyển sang dataframe
scaled_df = pd.DataFrame(scaled_features, columns=scaler.get_feature_names_out(continuous_columns))

# Kết hợp với bộ dữ liệu gốc
scaled_data = pd.concat([data.drop(columns=continuous_columns), scaled_df], axis=1)

Việc chuẩn hóa dữ liệu rất quan trọng để xác định ranh giới quyết định giữa các lớp tốt hơn bằng cách đảm bảo rằng các biến đặc trưng có độ biến thiên trên cùng một thang đo. Dữ liệu hiện đã sẵn sàng để được sử dụng cho bước huấn luyện và kiểm thử mô hình


### One-hot encoding
Chuyển đổi các biến phân loại sang dạng số bằng phương pháp mã hóa one-hot.


In [ ]:
# Xác định các cột phân loại.
categorical_columns = scaled_data.select_dtypes(include=['object']).columns.tolist()
categorical_columns.remove('NObeyesdad')  # Exclude target column

# Applying one-hot encoding
encoder = OneHotEncoder(sparse_output=False, drop='first')
encoded_features = encoder.fit_transform(scaled_data[categorical_columns])

# chuyển sang dataframe
encoded_df = pd.DataFrame(encoded_features, columns=encoder.get_feature_names_out(categorical_columns))

# Kết hợp với tập dữ liệu gốc
prepped_data = pd.concat([scaled_data.drop(columns=categorical_columns), encoded_df], axis=1)

Ta thấy rằng tất cả các biến phân loại hiện đã được chuyển thành các đặc trưng mã hóa one-hot. Điều này làm tăng tổng số trường lên 24


### Encode the target variable


In [ ]:
# Mã hóa target variable
prepped_data['NObeyesdad'] = prepped_data['NObeyesdad'].astype('category').cat.codes
prepped_data.head()

### Separate the input and target data


In [ ]:
# Preparing final dataset
X = prepped_data.drop('NObeyesdad', axis=1)
y = prepped_data['NObeyesdad']

## Model training and evaluation


### Splitting the data set
Chia tập train test 80/20


In [ ]:
# Splitting data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

### Logistic Regression with One-vs-All
Trong phương pháp One-vs-All (OvA):

Thuật toán huấn luyện một bộ phân loại nhị phân cho mỗi lớp.

Mỗi bộ phân loại học cách phân biệt một lớp duy nhất với tất cả các lớp còn lại gộp chung.

Nếu có k lớp, thì sẽ huấn luyện k bộ phân loại.

Khi dự đoán, thuật toán đánh giá tất cả các bộ phân loại trên mỗi mẫu đầu vào và chọn lớp có điểm tự tin (confidence score) cao nhất làm kết quả dự đoán.

Ưu điểm:

Đơn giản và hiệu quả hơn về số lượng bộ phân loại (k).

Dễ triển khai đối với các thuật toán vốn cung cấp điểm tự tin tự nhiên (như logistic regression, SVM).

Nhược điểm:

Các bộ phân loại có thể gặp khó khăn với mất cân bằng lớp, vì mỗi mô hình phải phân biệt một lớp nhỏ với tất cả các lớp còn lại gộp lại.

Đòi hỏi mô hình phải hoạt động tốt ngay cả khi dữ liệu bất cân bằng nghiêm trọng, do nhóm “all” thường có số mẫu lớn hơn nhiều so với nhóm “one”.

Train a logistic regression model using the One-vs-All strategy and evaluate its performance.


In [ ]:
# Training logistic regression model using One-vs-All (default)
model_ova = LogisticRegression(multi_class='ovr', max_iter=1000)
model_ova.fit(X_train, y_train)

Đánh giá trên tập test


In [ ]:
# Predictions
y_pred_ova = model_ova.predict(X_test)

# Evaluation metrics for OvA
print("One-vs-All (OvA) Strategy")
print(f"Accuracy: {np.round(100*accuracy_score(y_test, y_pred_ova),2)}%")

### Logistic Regression with OvO

Trong phương pháp One-vs-One (OvO):

Thuật toán huấn luyện một bộ phân loại nhị phân cho từng cặp lớp trong bộ dữ liệu.

Nếu có k lớp, tổng số bộ phân loại được huấn luyện sẽ là:
* $k(k-1)/2$

Mỗi bộ phân loại được huấn luyện để phân biệt hai lớp cụ thể, bỏ qua tất cả các lớp còn lại.

Khi dự đoán, tất cả các bộ phân loại sẽ được sử dụng và kết quả cuối cùng được xác định bằng cơ chế bỏ phiếu, chọn lớp thắng nhiều lần nhất trong các cặp so sánh.

Ưu điểm:

Phù hợp với các thuật toán có chi phí huấn luyện cao, vì mỗi bộ phân loại chỉ làm việc với một phần nhỏ của dữ liệu (chỉ gồm hai lớp).

Trong nhiều trường hợp có thể chính xác hơn, vì mỗi bộ phân loại chỉ tập trung vào việc phân biệt một cặp lớp duy nhất.

Nhược điểm:

Tốn kém về tính toán khi số lượng lớp lớn do số mô hình phải huấn luyện tăng rất nhanh.

Có thể dẫn đến dự đoán mơ hồ nếu kết quả bỏ phiếu bị hòa (tie).

Train a logistic regression model using the One-vs-One (OvO) strategy and evaluate its performance.


In [ ]:
# Training logistic regression model using One-vs-One
model_ovo = OneVsOneClassifier(LogisticRegression(max_iter=1000))
model_ovo.fit(X_train, y_train)

Đánh giá trên tập test


In [ ]:
# Predictions
y_pred_ovo = model_ovo.predict(X_test)

# Evaluation metrics for OvO
print("One-vs-One (OvO) Strategy")
print(f"Accuracy: {np.round(100*accuracy_score(y_test, y_pred_ovo),2)}%")

### Exercises


Q1. Thử nghiệm với các tỷ lệ test khác nhau trong hàm train_test_split (ví dụ: 0.1, 0.3) và quan sát ảnh hưởng của chúng đến hiệu suất mô hình


In [ ]:
# your code here
print("\n--- Running pipeline with test_size = 0.1 ---")
obesity_risk_pipeline(file_path, test_size=0.1)

print("\n--- Running pipeline with test_size = 0.3 ---")
obesity_risk_pipeline(file_path, test_size=0.3)

## Summary:

### Q&A
1.  **How do different `test_size` values (0.1 and 0.3) affect the performance of the One-vs-All (OvA) model?**
    When the `test_size` increased from 0.1 to 0.3, the OvA model's accuracy slightly decreased from 75.94% to 74.92%.

2.  **How do different `test_size` values (0.1 and 0.3) affect the performance of the One-vs-One (OvO) model?**
    When the `test_size` increased from 0.1 to 0.3, the OvO model's accuracy slightly increased from 90.57% to 90.85%.

3.  **What insights can be drawn regarding the impact of `test_size` on accuracy for both models?**
    The `test_size` had a minor impact on the accuracy of both models. The OvO strategy consistently outperformed the OvA strategy by a significant margin, indicating that the choice of multi-class strategy had a much more substantial effect on performance than the proportion of data allocated for testing.

### Data Analysis Key Findings
*   For a `test_size` of 0.1, the One-vs-All (OvA) model achieved an accuracy of 75.94%, while the One-vs-One (OvO) model achieved 90.57%.
*   For a `test_size` of 0.3, the One-vs-All (OvA) model's accuracy slightly decreased to 74.92%, and the One-vs-One (OvO) model's accuracy slightly increased to 90.85%.
*   The OvO strategy consistently showed significantly higher accuracy (over 90%) compared to the OvA strategy (around 75%) across both tested `test_size` values.
*   The change in `test_size` from 0.1 to 0.3 resulted in a minor decrease of approximately 1.02 percentage points for OvA accuracy and a minor increase of approximately 0.28 percentage points for OvO accuracy.

### Insights or Next Steps
*   The choice of multi-class strategy (OvO vs. OvA) has a far greater impact on model performance for this dataset than the specific `test_size` within the tested range.
*   Given the superior performance of the OvO strategy, further model optimization efforts should prioritize models using this approach for the obesity risk prediction task.


### Summary of Observations on `test_size` Impact

After running the `obesity_risk_pipeline` function with `test_size` values of 0.1 and 0.3, we can observe the following regarding the accuracy of the One-vs-All (OvA) and One-vs-One (OvO) models:

**For `test_size = 0.1` (10% test data):**
- **One-vs-All (OvA) Accuracy:** 75.94%
- **One-vs-One (OvO) Accuracy:** 90.57%

**For `test_size = 0.3` (30% test data):**
- **One-vs-All (OvA) Accuracy:** 74.92%
- **One-vs-One (OvO) Accuracy:** 90.85%

**Observations:**

1.  **One-vs-All (OvA) Model:** The accuracy slightly decreased from 75.94% to 74.92% when the `test_size` increased from 0.1 to 0.3. This suggests that with a larger test set, the model's generalization performance was slightly lower, possibly due to a smaller training set impacting the model's ability to learn complex patterns, or simply reflecting a more robust evaluation on a larger, more representative test sample.

2.  **One-vs-One (OvO) Model:** The accuracy slightly increased from 90.57% to 90.85% when the `test_size` increased from 0.1 to 0.3. This marginal improvement might indicate that the OvO strategy benefits from a larger, more diverse training set (as the training set size decreases from 90% to 70% of the total data, which is contradictory to expectation if the training set became smaller. However, the change is very small and could be due to random fluctuations in the `train_test_split` with `random_state=42` picking a slightly 'easier' 30% test set).

3.  **Comparison between OvA and OvO:** In both scenarios (test_size 0.1 and 0.3), the One-vs-One (OvO) strategy consistently outperformed the One-vs-All (OvA) strategy significantly in terms of accuracy. This highlights OvO's effectiveness for this particular multi-class classification problem and dataset.

In conclusion, while the `test_size` had a minor impact on the accuracy, the choice of multi-class strategy (OvO vs. OvA) had a much more substantial effect on the model's performance.

Q2. Vẽ biểu đồ cột (bar chart) thể hiện mức độ quan trọng của các đặc trưng bằng cách sử dụng các hệ số (coefficients) từ mô hình hồi quy logistic One-vs-All. Đồng thời, hãy thử thực hiện điều này cho mô hình One-vs-One.


In [ ]:

# Feature importance
feature_importance = np.mean(np.abs(model_ova.coef_), axis=0)
plt.barh(X.columns, feature_importance)
# your code here

Q3. Hãy viết một hàm obesity_risk_pipeline để tự động hóa toàn bộ quy trình:

1. Tải và tiền xử lý dữ liệu

2. Huấn luyện mô hình

3. Đánh giá mô hình

Hàm này cần nhận đường dẫn tệp (file path) và kích thước tập kiểm thử (test set size) làm tham số đầu vào.


In [ ]:
# write your function here and then execute this cell
def obesity_risk_pipeline(data_path, test_size=0.2):
    # 1. Load the data
    data = pd.read_csv(data_path)

    # 2. Preprocessing
    # Feature scaling for continuous columns
    continuous_columns = data.select_dtypes(include=['float64']).columns.tolist()
    scaler = StandardScaler()
    scaled_features = scaler.fit_transform(data[continuous_columns])
    scaled_df = pd.DataFrame(scaled_features, columns=scaler.get_feature_names_out(continuous_columns))
    scaled_data = pd.concat([data.drop(columns=continuous_columns), scaled_df], axis=1)

    # One-hot encoding for categorical columns
    categorical_columns = scaled_data.select_dtypes(include=['object']).columns.tolist()
    categorical_columns.remove('NObeyesdad')  # Exclude target column
    encoder = OneHotEncoder(sparse_output=False, drop='first')
    encoded_features = encoder.fit_transform(scaled_data[categorical_columns])
    encoded_df = pd.DataFrame(encoded_features, columns=encoder.get_feature_names_out(categorical_columns))
    prepped_data = pd.concat([scaled_data.drop(columns=categorical_columns), encoded_df], axis=1)

    # Encode the target variable
    prepped_data['NObeyesdad'] = prepped_data['NObeyesdad'].astype('category').cat.codes

    # Separate input and target data
    X = prepped_data.drop('NObeyesdad', axis=1)
    y = prepped_data['NObeyesdad']

    # Splitting the data set
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=42, stratify=y)

    # 3. Model Training and Evaluation

    # Logistic Regression with One-vs-All (OvA)
    model_ova = LogisticRegression(multi_class='ovr', max_iter=1000)
    model_ova.fit(X_train, y_train)
    y_pred_ova = model_ova.predict(X_test)
    print(f"One-vs-All (OvA) Strategy with test size {test_size*100}%:")
    print(f"Accuracy: {np.round(100*accuracy_score(y_test, y_pred_ova),2)}%")

    # Logistic Regression with One-vs-One (OvO)
    model_ovo = OneVsOneClassifier(LogisticRegression(max_iter=1000))
    model_ovo.fit(X_train, y_train)
    y_pred_ovo = model_ovo.predict(X_test)
    print(f"\nOne-vs-One (OvO) Strategy with test size {test_size*100}%:")
    print(f"Accuracy: {np.round(100*accuracy_score(y_test, y_pred_ovo),2)}%")

obesity_risk_pipeline(file_path, test_size=0.2)

## Ghi chú học tập / Study prompts

- Đầu vào của từng hàm là gì, đầu ra có kiểu và kích thước nào?
- Thay đổi tham số hoặc dữ liệu nhỏ sẽ ảnh hưởng kết quả ra sao?
- Cell nào cần dữ liệu bên ngoài, thư viện hoặc thao tác nhập tay?

Explain the data flow, modify one example, and record the result when revisiting this exercise.